# NAKO to BIDS Conversion

In [2]:
# run_nako_batch.py
import os
import glob
from pathlib import Path
from tqdm import tqdm
from nako_bids import process_single_nako_subject_from_zip

# Configure your NAKO root directory
NAKO_ROOT = r"NAKO-1048_MRT"

# Locate all resting-state ZIP files
zip_folder = os.path.join(NAKO_ROOT, "Resting_State_TRA")
zip_files = sorted(glob.glob(os.path.join(zip_folder, "*_Resting_State_TRA.zip")))

if not zip_files:
    raise FileNotFoundError(f"No ZIP files found in: {zip_folder}")

print(f"\n🔍 Found {len(zip_files)} subjects. Starting batch processing...\n")

failed_subjects = []

for zip_path in tqdm(zip_files, desc="Processing subjects", unit="subject"):
    stem = Path(zip_path).stem
    if stem.endswith("_Resting_State_TRA"):
        base_id = "_".join(stem.split("_")[:2])
        
        # Optional: Skip if already processed (checks for bold.nii.gz)
        bids_func_dir = os.path.join(NAKO_ROOT, "BIDS", f"sub-{base_id.replace('_', '')}", "func")
        bold_file = os.path.join(bids_func_dir, f"sub-{base_id.replace('_', '')}_task-rest_bold.nii.gz")
        if os.path.exists(bold_file):
            continue  # Already done — skip to save time

        try:
            process_single_nako_subject_from_zip(base_id, NAKO_ROOT)
        except Exception as e:
            # Log the error and continue
            print(f"\n⚠️ Subject {base_id} failed: {str(e)}")
            failed_subjects.append(base_id)

# Final summary
print(f"\n{'='*60}")
print(f"✅ Batch processing complete.")
print(f"   Total subjects: {len(zip_files)}")
print(f"   Successfully processed: {len(zip_files) - len(failed_subjects)}")
print(f"   Failed subjects: {len(failed_subjects)}")
if failed_subjects:
    print("   Failed IDs:", failed_subjects)
else:
    print("   All subjects processed successfully.")


🔍 Found 1 subjects. Starting batch processing...



Processing subjects:   0%|          | 0/1 [00:00<?, ?subject/s]


=== Step 1: Reconstructing fMRI from Siemens mosaic DICOMs ===
✅ Reconstructed 180 volumes. Shape: (64, 64, 36, 180)

=== Step 2: Writing BIDS structure ===

✅ Successfully processed NAKO subject 13091130 into BIDS at: NAKO-1048_MRT\BIDS
ℹ️ PhaseEncodingDirection: i-
ℹ️ TotalReadoutTime: 0.035000 s


Processing subjects: 100%|██████████| 1/1 [00:09<00:00,  9.99s/subject]


✅ Batch processing complete.
   Total subjects: 1
   Successfully processed: 1
   Failed subjects: 0
   All subjects processed successfully.
